# Datastores

In [1]:
import dotenv
dotenv.load_dotenv(override=True)

True

Automated setup of a dummy storage account to play arround with.

Get the storage account key and the SAS token from the storage account. Because we cannot set environment variables with Jupyter %% magic, we need to update the .env file with the variables.

In [2]:
%%writefile ./helper_code/create_a_blob_store.sh

STORAGE_ACCOUNT_NAME='dg123'
STORAGE_ACCOUNT_CONTAINER_NAME='csvdata'

echo $(pwd)

# Check if container exists. If yes, delete. 
if az storage account show --name $STORAGE_ACCOUNT_NAME &>/dev/null; then
    echo "storage account with name ${STORAGE_ACCOUNT_NAME} already exists"
    echo '... start deleting'
    az storage account delete --name $STORAGE_ACCOUNT_NAME --yes
fi

# Create the azure storage account
echo "... Start creating storage account"
az storage account create --name $STORAGE_ACCOUNT_NAME --resource-group $AZURE_RESOURCE_GROUP \
    --kind BlobStorage \
    --location $AZURE_LOCATION \
    --access-tier Hot \
    --sku Standard_LRS &>/dev/null

# Get the account key
echo "... Get the account key"
ACCOUNT_KEY=$(az storage account keys list --account-name $STORAGE_ACCOUNT_NAME --query '[0].value' -o tsv)

# Create the container
echo "... Create a container"
az storage container create --name csvdata \
    --account-key $ACCOUNT_KEY \
    --account-name $STORAGE_ACCOUNT_NAME &>/dev/null

# Upload a CSV
echo "... Upload a CSV"
az storage blob upload \
    --account-name $STORAGE_ACCOUNT_NAME \
    --account-key $ACCOUNT_KEY \
    --container-name $STORAGE_ACCOUNT_CONTAINER_NAME \
    --name "diabetes.csv" \
    --file "../../data/diabetes-data/diabetes.csv" &>/dev/null


# Create the SAS token
echo "... Create an SAS token"
SAS_TOKEN=$(az storage container generate-sas \
    --account-key=$ACCOUNT_KEY \
    --account-name dg123 \
    --name csvdata \
    --permissions rld \
    --expiry 2026-12-31T23:59:59Z \
    --output tsv)

# Create a new .env file
echo "... Create a new .env file"
CONTENT_DOTENV=$(cat ../../.env | grep -v AZURE_STORAGE_ACCOUNT | grep -v AZURE_SAS_TOKEN)
echo -e "${CONTENT_DOTENV}\nAZURE_STORAGE_ACCOUNT=${ACCOUNT_KEY}\nAZURE_SAS_TOKEN=${SAS_TOKEN}" > ../../.env

echo 'Done!'

Overwriting ./helper_code/create_a_blob_store.sh


In [3]:
!sh ./helper_code/create_a_blob_store.sh

/c/Users/dgouwy/Documents/devoProjects/azure-machine-learning/az-ml/03_data-in-az-ml
storage account with name dg123 already exists
... start deleting
... Start creating storage account
... Get the account key
... Create a container
... Upload a CSV
... Create an SAS token
... Create a new .env file
Done!


Continue as per usual.

In [4]:
from azure.ai.ml.entities import AzureBlobDatastore, AccountKeyConfiguration, SasTokenConfiguration
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import os
import dotenv
import uuid

dotenv.load_dotenv(override=True)

ml_client = MLClient(
     credential         = DefaultAzureCredential()
    ,subscription_id    = os.environ["AZURE_SUBSCRIPTION_ID"]
    ,resource_group_name= os.environ["AZURE_RESOURCE_GROUP"]
    ,workspace_name     = os.environ["AZURE_ML"]
)

storage_account = "dg123"
container_name = "csvdata"

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [5]:
%%sh
printenv | grep 'AZURE' | cut -d '=' -f1 | awk '{print $0"=*******"}'

AZURE_RESOURCE_GROUP=*******
AZURE_ML=*******
AZURE_SUBSCRIPTION_ID=*******
AZURE_LOCATION=*******
AZURE_SAS_TOKEN=*******
AZURE_STORAGE_ACCOUNT=*******
AZUREML_CURRENT_CLOUD=*******


## Setup a data store pointing to a storage account container

(Renew the tokens in the .env file if the data store has been recreated.)

Example of connecting through an account key:

In [6]:
blob_datastore = AzureBlobDatastore(
    			name = "csv_container",
    			description = "Datastore pointing to a blob container",
    			account_name = storage_account,
    			container_name = container_name,
    			credentials = AccountKeyConfiguration(
        			account_key=os.environ["AZURE_STORAGE_ACCOUNT"]
    			),
)

blob_datastore_info = ml_client.create_or_update(blob_datastore)

Example of connecting through an SAS (shared access signature) token:

In [7]:
data_store_name = "blob_sas_example"

blob_datastore = AzureBlobDatastore(
		name			= data_store_name,
		description		= "Datastore pointing to a blob container",
		account_name	= storage_account,
		container_name	= container_name,
		credentials		= SasTokenConfiguration(
									sas_token = os.environ["AZURE_SAS_TOKEN"]
									)
)

blob_datastore_info = ml_client.create_or_update(blob_datastore)

In [8]:
[ds.name for ds in ml_client.datastores.list()]

['blob_sas_example',
 'csv_container',
 'azureml_globaldatasets',
 'workspaceartifactstore',
 'workspaceblobstore',
 'workspaceworkingdirectory',
 'workspacefilestore']

In [9]:
try:
    ml_client.datastores.delete("blob_sas_example")
except:
    pass

try:
    ml_client.datastores.delete("csv_container")
except:
    pass

In [10]:
!az storage account delete --name dg123 --yes